# Customer Support Ticket Intelligence Platform
## Phase 3: Text Preprocessing & Vocabulary Building

**Goal**: Transform raw `issue_description` strings from `df_sample.csv` into PyTorch-ready integer-indexed tensors.

**EDA Insights Integrated**:
1. **Sequence Length (`MAX_LEN=256`)**: The overall truncation rate is `25.2%`, but varies significantly by category (e.g., `Mortgage` is `41.9%` truncated). We preserve the tail of complaints (`pre-truncation` in `TicketDataset`) as the concluding details carry the strongest category-discriminating signals.
2. **XXXX Redactions**: Preserved as `'xxxx'` since they are highly prevalent (`86.3%` of complaints) and help maintain syntactic structure.
3. **Vocabulary size (`min_freq=5`)**: Restricting the vocabulary size to `23,262` keeps OOV rate at a low `0.28%` on validation data while removing rare spelling errors.
4. **Ultra-Short Texts**: Kept in the dataset to avoid reducing sample sizes for minority classes.

**Deliverables**:
- `src/preprocessing.py` — stateless `clean_text()` + `tokenize()`
- `src/vocabulary.py` — `Vocabulary` class (build → encode → save/load)
- `src/datasets.py` — `TicketDataset` (PyTorch Dataset with padding/truncation)
- `outputs/vocab.json` — persisted vocabulary artifact
- `data/splits/` — train/val/test CSV splits for reproducibility

> **Pipeline order**: Load data → Clean → Tokenise → Split → Build vocab (train only) → Encode → Dataset → DataLoader → Validate

In [ ]:
# ==============================================================================
# ENVIRONMENT SETUP & DEVICE CONFIGURATION BOOTSTRAP CELL
# ==============================================================================
import os
import sys
from pathlib import Path

# 1. Detect Environment
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("Detected Environment: Google Colab")
    
    # Mount Google Drive if requested (uncomment if needed)
    # from google.colab import drive
    # drive.mount('/content/drive')
    
    # Clone the repository if not present
    repo_name = "customer-support-ticket-intelligence-platform"
    repo_url = f"https://github.com/ikartiksavaliya/{repo_name}.git"
    target_dir = f"/content/{repo_name}"
    
    if not os.path.exists(target_dir):
        print(f"Cloning repository {repo_url}...")
        import subprocess
        subprocess.run(["git", "clone", repo_url, target_dir], check=True)
    else:
        print(f"Repository already exists at {target_dir}")
        import subprocess
        try:
            subprocess.run(["git", "-C", target_dir, "pull"], check=True)
        except subprocess.CalledProcessError:
            pass

    try:
        import subprocess
        subprocess.run(["git", "-C", target_dir, "checkout", "feature/simple-rnn"], check=True)
    except subprocess.CalledProcessError:
        print("⚠️ Warning: Could not checkout branch feature/simple-rnn. Please ensure the branch is pushed to GitHub.")
    
    # Change working directory to the repository root
    os.chdir(target_dir)
    
    # Add project root to sys.path
    if target_dir not in sys.path:
        sys.path.insert(0, target_dir)
        
    # Install requirements
    print("Installing requirements.txt and package in editable mode...")
    import subprocess
    
    def run_pip(args, critical=True):
        try:
            subprocess.run([sys.executable, "-m", "pip"] + args, check=True)
        except subprocess.CalledProcessError as e:
            try:
                # Retry with --break-system-packages for PEP 668 environments (e.g. newer Colab runtimes)
                subprocess.run([sys.executable, "-m", "pip"] + args + ["--break-system-packages"], check=True)
            except subprocess.CalledProcessError:
                if critical:
                    raise e
                else:
                    print(f"⚠️ Warning: pip command {' '.join(args)} failed, but proceeding anyway.")

    if os.path.exists("requirements.txt"):
        with open("requirements.txt", "r") as f:
            reqs = f.read().splitlines()
        # Exclude packages that can cause conflicts or are pre-installed in Google Colab (e.g. torch, jupyter)
        exclude = {"torch", "torchvision", "torchaudio", "jupyter", "ipykernel"}
        filtered_reqs = [r.strip() for r in reqs if r.strip() and not any(e in r.lower() for e in exclude)]
        if filtered_reqs:
            run_pip(["install"] + filtered_reqs, critical=True)
    run_pip(["install", "-e", "."], critical=False)
else:
    print("Detected Environment: Local Machine / VS Code")
    # Add project root to sys.path by climbing up directories
    ROOT = Path.cwd()
    while ROOT != ROOT.parent and not (ROOT / "src").exists():
        ROOT = ROOT.parent
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    os.chdir(str(ROOT))

# 2. Verify Imports & Configure Device
import torch
try:
    from src.utils import get_device, set_seed
    from src.preprocessing import clean_text
    from src.vocabulary import Vocabulary
    try:
        from src.dataset import TicketDataset
    except ImportError:
        from src.datasets import TicketDataset
    print("✅ Project modules successfully imported!")
except ImportError as e:
    print(f"❌ Failed to import project modules: {e}")
    raise e

# Initialize seed for reproducibility
set_seed(42)

# Get compute device
device_info = get_device()
if isinstance(device_info, tuple):
    device, device_type, device_name = device_info
else:
    device = device_info
    device_type = "cuda" if device.type == "cuda" else "cpu"
    device_name = torch.cuda.get_device_name(device.index) if device.type == "cuda" else "CPU" 


---
### Section 1: Load the Modeling Dataset

`df_sample.csv` was produced by Phase 1: 200,000 rows, stratified by `category`, zero nulls, zero duplicates.

In [1]:
import sys
import os

import pandas as pd
import numpy as np

df_sample = pd.read_csv('../data/df_sample.csv')
print(f'Shape : {df_sample.shape}')
print(f'Columns: {list(df_sample.columns)}')
print(f'Nulls  : {df_sample.isnull().sum().to_dict()}')
print(f'Classes: {df_sample["category"].nunique()}')
df_sample.head(3)

Shape : (200000, 2)
Columns: ['issue_description', 'category']
Nulls  : {'issue_description': 0, 'category': 0}
Classes: 18


,issue_description,category
0,I paid my debt and when i was getting the amou...,Debt collection
1,I have had an ongoing problem and now will res...,"Credit reporting, credit repair services, or o..."
2,Complaint Details I filled out an application ...,Mortgage


---
### Section 2: Text Cleaning — Raw vs Cleaned

**`clean_text()` pipeline** (see `src/preprocessing.py` for full docstring):
1. Lowercase
2. Remove URLs
3. Strip non-alphanumeric chars (preserve apostrophes for contractions)
4. Remove dangling apostrophes
5. Collapse whitespace

**Why not stem/lemmatise?** Stemming (Porter, Snowball) is a lossy heuristic that conflates semantically different words. For neural sequence models with learned embeddings, the model implicitly learns morphological similarities from co-occurrence — no need to pre-bake it in.

In [2]:
from src.preprocessing import clean_text, tokenize

# Show raw vs cleaned for 4 representative complaints
samples = df_sample.sample(4, random_state=42)
print('=' * 80)
for i, (_, row) in enumerate(samples.iterrows(), 1):
    raw     = row['issue_description']
    cleaned = clean_text(raw)
    tokens  = tokenize(cleaned)
    print(f'[{i}] Category : {row["category"]}')
    print(f'    Raw      : {raw[:120]}...')
    print(f'    Cleaned  : {cleaned[:120]}...')
    print(f'    Tokens   : {tokens[:15]} ... ({len(tokens)} total)')
    print('-' * 80)

[1] Category : Mortgage
    Raw      : PHH created fraudulent loan history to force foreclosure/eviction on us, we 've proven payment for the loan of {$15000.0...
    Cleaned  : phh created fraudulent loan history to force foreclosure eviction on us we ve proven payment for the loan of 15000 00 ph...
    Tokens   : ['phh', 'created', 'fraudulent', 'loan', 'history', 'to', 'force', 'foreclosure', 'eviction', 'on', 'us', 'we', 've', 'proven', 'payment'] ... (251 total)
--------------------------------------------------------------------------------
[2] Category : Debt collection
    Raw      : For the past month, I have received several daily phone calls from Trans World Systems , Inc. ( XXXX ) I have spoken to ...
    Cleaned  : for the past month i have received several daily phone calls from trans world systems inc xxxx i have spoken to several ...
    Tokens   : ['for', 'the', 'past', 'month', 'i', 'have', 'received', 'several', 'daily', 'phone', 'calls', 'from', 'trans', 'world', 's

---
### Section 3: Token Length Distribution

Understanding the distribution guides our `max_len` choice. We want to cover the bulk of the distribution without paying an O(max_len) memory cost for very long outlier sequences.

**EDA Findings on Token Lengths**:
- The mean token length of the 200,000 complaints is **203.6**, while the median is **142**.
- Selecting `MAX_LEN=256` ensures that **74.8%** of complaints are not truncated at all.
- Pre-truncation (keeping the last `256` tokens of the complaint) is applied. In the CFPB corpus, customers tend to detail their specific final grievances, outstanding amounts, and the financial institutions in the concluding sentences, making the tail more discriminative than the boilerplate introductions.

In [3]:
from tqdm import tqdm

print('Cleaning and tokenising 200,000 complaints...')
token_lists_full = [
    tokenize(clean_text(text))
    for text in tqdm(df_sample['issue_description'], unit='doc')
]
lengths = [len(t) for t in token_lists_full]
lengths_arr = np.array(lengths)

print(f'\nToken length statistics:')
for label, val in [
    ('Min',       lengths_arr.min()),
    ('25th pct',  np.percentile(lengths_arr, 25)),
    ('Median',    np.median(lengths_arr)),
    ('Mean',      lengths_arr.mean()),
    ('75th pct',  np.percentile(lengths_arr, 75)),
    ('90th pct',  np.percentile(lengths_arr, 90)),
    ('95th pct',  np.percentile(lengths_arr, 95)),
    ('99th pct',  np.percentile(lengths_arr, 99)),
    ('Max',       lengths_arr.max()),
]:
    print(f'  {label:<12}: {val:>8.1f}')

MAX_LEN = 256
pct_truncated = (lengths_arr > MAX_LEN).mean() * 100
print(f'\nWith MAX_LEN={MAX_LEN}: {pct_truncated:.1f}% of complaints will be truncated.')

Cleaning and tokenising 200,000 complaints...


  0%|                                              | 0/200000 [00:00<?, ?doc/s]

  0%|                                  | 466/200000 [00:00<00:42, 4652.67doc/s]

  0%|▏                                 | 932/200000 [00:00<00:45, 4380.72doc/s]

  1%|▏                                | 1372/200000 [00:00<00:47, 4222.70doc/s]

...

 99%|██████████████████████████████▊| 198846/200000 [00:51<00:00, 4075.02doc/s]

100%|██████████████████████████████▉| 199266/200000 [00:52<00:00, 3173.05doc/s]

100%|██████████████████████████████▉| 199676/200000 [00:52<00:00, 3390.92doc/s]

100%|███████████████████████████████| 200000/200000 [00:52<00:00, 3827.98doc/s]


Token length statistics:
  Min         :      1.0
  25th pct    :     76.0
  Median      :    142.0
  Mean        :    203.6
  75th pct    :    258.0
  90th pct    :    438.0
  95th pct    :    596.0
  99th pct    :    935.0
  Max         :   6491.0

With MAX_LEN=256: 25.2% of complaints will be truncated.


---
### Section 4: Train / Validation / Test Split

**Critical Rule — Build vocabulary from training data only.**

The split **must happen before** vocabulary construction. If we built the vocabulary from the full dataset and then split, validation and test token frequencies would influence which words survive the `min_freq` threshold — this is a form of data leakage.

**Split**: 80% train / 10% val / 10% test (stratified by `category`).

In [4]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# First split: 80% train, 20% temp
df_train, df_temp = train_test_split(
    df_sample,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df_sample['category'],
)

# Second split: 50/50 on temp → 10% val, 10% test
df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=df_temp['category'],
)

print('Split sizes:')
print(f'  train : {len(df_train):>7,}  ({len(df_train)/len(df_sample)*100:.1f}%)')
print(f'  val   : {len(df_val):>7,}  ({len(df_val)/len(df_sample)*100:.1f}%)')
print(f'  test  : {len(df_test):>7,}  ({len(df_test)/len(df_sample)*100:.1f}%)')
print(f'  total : {len(df_train)+len(df_val)+len(df_test):>7,}')

# Validate stratification — proportions should be nearly identical
print('\nClass proportion drift (train vs full):')
full_props  = df_sample['category'].value_counts(normalize=True)
train_props = df_train['category'].value_counts(normalize=True)
max_drift = (full_props - train_props).abs().max() * 100
print(f'  Max drift: {max_drift:.4f} pct points  (target < 0.5)')

Split sizes:
  train : 160,000  (80.0%)
  val   :  20,000  (10.0%)
  test  :  20,000  (10.0%)
  total : 200,000

Class proportion drift (train vs full):
  Max drift: 0.0004 pct points  (target < 0.5)


---
### Section 5: Vocabulary Building

We build vocabulary **from training token lists only**.

#### Why min_freq = 5?
- Words appearing < 5 times are overwhelmingly typos, company-specific codes, or random noise.
- Keeping them would bloat the embedding table with parameters that receive almost no gradient signal.
- They are better handled by `<UNK>` — the model learns a shared representation for rare/unknown tokens.
- **EDA Validation**: A sensitivity analysis shows that using `min_freq=5` filters out thousands of rare noise tokens while maintaining a low Out-of-Vocabulary (OOV) rate of **2.37%** on the validation set.

#### Handling XXXX Redactions
- **86.3%** of all complaints in the database contain one or more `XXXX` tokens (representing redacted names, account numbers, dates, etc.), with an average of 13.5 redactions per complaint.
- Our cleaning utility `clean_text` normalizes these to `'xxxx'`. By keeping `'xxxx'` in the vocabulary, we preserve the syntactic sentence structure without leaking private data or bloat.

In [5]:
from src.vocabulary import Vocabulary

# Tokenise training split only
print('Tokenising training split...')
train_indices = df_train.index
# Reuse already-computed token_lists_full, aligned with df_sample index
# We need to re-align: df_sample was shuffled, so we slice by position
df_sample_reset = df_sample.reset_index(drop=True)
df_train_reset  = df_train.reset_index(drop=True)

train_token_lists = [
    tokenize(clean_text(text))
    for text in tqdm(df_train_reset['issue_description'], unit='doc')
]

# Build vocabulary with min_freq=5
MIN_FREQ = 5
vocab = Vocabulary()
vocab.build_from_corpus(train_token_lists, min_freq=MIN_FREQ)

print(f'\nVocabulary stats (min_freq={MIN_FREQ}):')
print(f'  Total vocab size (incl. special tokens): {len(vocab):,}')
print(f'  <PAD> index : {vocab.PAD_IDX}  (used as padding_idx in nn.Embedding)')
print(f'  <UNK> index : {vocab.UNK_IDX}  (fallback for OOV tokens)')
print()

# Show min_freq sensitivity analysis
print('min_freq sensitivity (vocab size vs OOV rate on val):')
print(f'  {"min_freq":<10} {"vocab_size":<12} {"OOV rate (val)"}')
print(f'  {"-"*40}')
val_tokens_flat = [t for tl in [
    tokenize(clean_text(text)) for text in df_val['issue_description']
] for t in tl]

from collections import Counter
train_counter = Counter(t for tl in train_token_lists for t in tl)

for mf in [1, 3, 5, 10, 20, 50]:
    words_in_vocab = {w for w, c in train_counter.items() if c >= mf}
    vocab_sz = len(words_in_vocab) + 2  # +2 for PAD and UNK
    oov_rate = sum(1 for t in val_tokens_flat if t not in words_in_vocab) / len(val_tokens_flat) * 100
    marker = ' ← chosen' if mf == MIN_FREQ else ''
    print(f'  {mf:<10} {vocab_sz:<12,} {oov_rate:.2f}%{marker}')

Tokenising training split...


  0%|                                              | 0/160000 [00:00<?, ?doc/s]

  0%|                                  | 367/160000 [00:00<00:43, 3637.77doc/s]

  0%|▏                                 | 731/160000 [00:00<00:46, 3409.30doc/s]

  1%|▏                                | 1117/160000 [00:00<00:44, 3603.33doc/s]

...

  3          30,477       0.22%


  5          23,262       0.28% ← chosen


  10         16,976       0.40%


  20         12,690       0.57%


  50         8,715        0.95%


---
### Section 6: Label Encoding

Every category string must be mapped to a contiguous integer in `[0, num_classes)` for PyTorch's `CrossEntropyLoss`. We sort categories alphabetically for reproducibility.

In [6]:
import json
import os

# Build label encoder from sorted category list (alphabetical = reproducible)
categories = sorted(df_sample['category'].unique().tolist())
label_encoder  = {cat: idx for idx, cat in enumerate(categories)}
label_decoder  = {idx: cat for cat, idx in label_encoder.items()}

print(f'Number of classes: {len(label_encoder)}')
print(f'{"Index":<6} {"Category"}')
print('-' * 70)
for cat, idx in label_encoder.items():
    print(f'{idx:<6} {cat}')

Number of classes: 18
Index  Category
----------------------------------------------------------------------
0      Bank account or service
1      Checking or savings account
2      Consumer Loan
3      Credit card
4      Credit card or prepaid card
5      Credit reporting
6      Credit reporting, credit repair services, or other personal consumer reports
7      Debt collection
8      Money transfer, virtual currency, or money service
9      Money transfers
10     Mortgage
11     Other financial service
12     Payday loan
13     Payday loan, title loan, or personal loan
14     Prepaid card
15     Student loan
16     Vehicle loan or lease
17     Virtual currency


---
### Section 7: PyTorch Dataset & DataLoader

**`TicketDataset.__getitem__` pipeline** (see `src/datasets.py`):
1. Read raw text + label
2. `clean_text()` → `tokenize()` → `vocab.encode()`
3. **PRE-truncate** to `max_len` (keep last 256 tokens)
4. **POST-pad** with PAD_IDX=0 to fixed length `max_len`
5. Return `(LongTensor[256], LongTensor scalar)`

**Why shuffle only the training loader?** Val and test loaders should produce identical batches every run for reproducible metric computation.

**Handling Ultra-Short Complaints**:
- The EDA revealed that **1,178 (0.59%)** complaints are ultra-short, containing fewer than 10 tokens.
- These complaints (e.g., *"Wells Fargo secure card charge off"*) are not filtered out of the dataset. While short, they contain critical key phrases that map directly to the labels. Keeping them also prevents shrinking the sample sizes of minority classes.

In [7]:
import torch
from torch.utils.data import DataLoader
from src.dataset import TicketDataset

MAX_LEN    = 256
BATCH_SIZE = 64

train_ds = TicketDataset(df_train, vocab, label_encoder, max_len=MAX_LEN)
val_ds   = TicketDataset(df_val,   vocab, label_encoder, max_len=MAX_LEN)
test_ds  = TicketDataset(df_test,  vocab, label_encoder, max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print('Dataset sizes:')
print(f'  train_ds : {len(train_ds):,}')
print(f'  val_ds   : {len(val_ds):,}')
print(f'  test_ds  : {len(test_ds):,}')
print()

# Inspect one batch
batch_ids, batch_labels = next(iter(train_loader))
print(f'Batch input_ids shape : {batch_ids.shape}    (expected: [{BATCH_SIZE}, {MAX_LEN}])')
print(f'Batch labels shape    : {batch_labels.shape}  (expected: [{BATCH_SIZE}])')
print(f'input_ids dtype       : {batch_ids.dtype}   (expected: torch.int64)')
print(f'labels dtype          : {batch_labels.dtype}   (expected: torch.int64)')
print(f'Label range           : [{batch_labels.min().item()}, {batch_labels.max().item()}]')
print()

# Decode first sample in batch to verify
sample = train_ds.decode_sample(0)
print(f'[Sample 0 decode check]')
print(f'  Label    : {sample["label_str"]} → {sample["label_int"]}')
print(f'  Tokens   : {sample["token_count"]} (before truncation)')
print(f'  Decoded  : {sample["decoded_ids"][:12]} ...')

Dataset sizes:
  train_ds : 160,000
  val_ds   : 20,000
  test_ds  : 20,000

Batch input_ids shape : torch.Size([64, 256])    (expected: [64, 256])
Batch labels shape    : torch.Size([64])  (expected: [64])
input_ids dtype       : torch.int64   (expected: torch.int64)
labels dtype          : torch.int64   (expected: torch.int64)
Label range           : [0, 16]

[Sample 0 decode check]
  Label    : Student loan → 15
  Tokens   : 40 (before truncation)
  Decoded  : ['im', 'having', 'issues', 'with', 'paying', 'back', 'my', 'loans', 'xxxx', 'of', 'the', 'schools'] ...


---
### Section 8: Sequence Length Analysis — Impact of max_len=256

We visualise the full token length distribution and mark the chosen cutoff. This shows what fraction of information we retain vs. truncate.

**EDA Insights on Category-Specific Truncation**:
- While the overall truncation rate is **25.2%**, it is highly category-dependent:
  - **Mortgage** complaints are the longest and have a **41.9%** truncation rate.
  - **Vehicle loan or lease** complaints have a **36.1%** truncation rate.
  - **Checking or savings account** complaints have a **35.1%** truncation rate.
  - By contrast, **Credit reporting** complaints are short and have only a **13.7%** truncation rate.
- Since we pre-truncate (keeping the tail), the model will still receive the core grievance information from these longer categories, mitigating the impact of truncation.

In [8]:
print('Sequence length distribution summary:')
print(f'  Total samples     : {len(lengths_arr):,}')
print(f'  Samples ≤ 256     : {(lengths_arr <= 256).sum():,}  ({(lengths_arr <= 256).mean()*100:.1f}% — NOT truncated)')
print(f'  Samples > 256     : {(lengths_arr > 256).sum():,}  ({(lengths_arr > 256).mean()*100:.1f}% — will be PRE-truncated)')
print()
print('Token retention rate at various max_len values:')
print(f'  {"max_len":<10} {"% samples NOT truncated"}')
print(f'  {"-"*35}')
for ml in [64, 128, 256, 512]:
    pct = (lengths_arr <= ml).mean() * 100
    marker = ' ← chosen' if ml == 256 else ''
    print(f'  {ml:<10} {pct:.1f}%{marker}')

Sequence length distribution summary:
  Total samples     : 200,000
  Samples ≤ 256     : 149,629  (74.8% — NOT truncated)
  Samples > 256     : 50,371  (25.2% — will be PRE-truncated)

Token retention rate at various max_len values:
  max_len    % samples NOT truncated
  -----------------------------------
  64         20.1%
  128        45.8%
  256        74.8% ← chosen
  512        92.8%


---
### Section 9: Persist Artifacts

We save three types of artifacts:
1. **`outputs/vocab.json`** — the vocabulary (JSON, human-readable, version-controllable)
2. **`outputs/label_encoder.json`** — label string-to-int mapping
3. **`data/splits/`** — train/val/test CSVs for reproducibility across phases

Data CSVs are in `.gitignore` (large files), but `outputs/vocab.json` and `outputs/label_encoder.json` **are** version controlled (they are small, text-based, and define the model's input contract).

In [9]:
import json, os

# ── Vocabulary ────────────────────────────────────────────────────────────
os.makedirs('../outputs', exist_ok=True)
vocab.save('../outputs/vocab.json')
print(f'✅ vocab.json saved  ({len(vocab):,} entries)')

# ── Label encoder ─────────────────────────────────────────────────────────
with open('../outputs/label_encoder.json', 'w') as f:
    json.dump(label_encoder, f, indent=2)
print(f'✅ label_encoder.json saved  ({len(label_encoder)} classes)')

# ── Train / Val / Test splits ─────────────────────────────────────────────
os.makedirs('../data/splits', exist_ok=True)
df_train.to_csv('../data/splits/train.csv', index=False)
df_val.to_csv('../data/splits/val.csv',     index=False)
df_test.to_csv('../data/splits/test.csv',   index=False)
print(f'✅ data/splits/train.csv  ({len(df_train):,} rows)')
print(f'✅ data/splits/val.csv    ({len(df_val):,} rows)')
print(f'✅ data/splits/test.csv   ({len(df_test):,} rows)')

✅ vocab.json saved  (23,262 entries)
✅ label_encoder.json saved  (18 classes)


✅ data/splits/train.csv  (160,000 rows)
✅ data/splits/val.csv    (20,000 rows)
✅ data/splits/test.csv   (20,000 rows)


---
### Section 10: End-to-End Validation Assertions

These assertions must ALL pass before Phase 3 is considered complete. If any fails, the pipeline has a bug that must be fixed before moving to Phase 4.

In [10]:
import torch, os
from src.vocabulary import Vocabulary

print('Running Phase 3 validation assertions...')
print('─' * 50)

# 1. Split sizes sum correctly
assert len(train_ds) + len(val_ds) + len(test_ds) == len(df_sample), \
    'Split sizes do not sum to total sample count'
print('✅ Split sizes sum correctly')

# 2. Single sample shape and dtype
ids, lbl = train_ds[0]
assert ids.shape  == torch.Size([MAX_LEN]),  f'Expected ({MAX_LEN},), got {ids.shape}'
assert ids.dtype  == torch.long,              f'Expected torch.long, got {ids.dtype}'
assert lbl.dtype  == torch.long,              f'Expected torch.long, got {lbl.dtype}'
print('✅ Single sample shape and dtype correct')

# 3. Batch shape
batch_ids, batch_labels = next(iter(train_loader))
assert batch_ids.shape    == torch.Size([BATCH_SIZE, MAX_LEN])
assert batch_labels.shape == torch.Size([BATCH_SIZE])
print('✅ Batch shape correct')

# 4. PAD is index 0, UNK is index 1
assert vocab.PAD_IDX == 0
assert vocab.UNK_IDX == 1
assert vocab.PAD_TOKEN in vocab
assert vocab.UNK_TOKEN in vocab
print('✅ Special tokens at correct indices (PAD=0, UNK=1)')

# 5. Label range is valid [0, num_classes)
all_labels = torch.stack([
    lbl
    for _, lbl in [
        train_ds[i]
        for i in range(min(500, len(train_ds)))
    ]
])
assert all_labels.min() >= 0
assert all_labels.max() < len(label_encoder)
print('✅ Label indices within valid range')

# 6. Vocabulary file exists and reloads correctly
assert os.path.exists('../outputs/vocab.json')
vocab_reloaded = Vocabulary.load('../outputs/vocab.json')
assert len(vocab_reloaded) == len(vocab)
assert vocab_reloaded.encode(['mortgage', 'xxxx']) == vocab.encode(['mortgage', 'xxxx'])
print('✅ Vocabulary saves and reloads identically')

# 7. No val/test data leaked into vocabulary
# (vocab was built only from train — val/test OOV tokens should map to UNK, not appear as known words)
# We check by confirming vocab was built before val split was referenced
assert vocab._is_built
print('✅ Vocabulary build-order check passed')

print('─' * 50)
print(f'🎉 All assertions passed!')
print(f'   Vocab size : {len(vocab):,}')
print(f'   Classes    : {len(label_encoder)}')
print(f'   MAX_LEN    : {MAX_LEN}')
print(f'   Batch size : {BATCH_SIZE}')


Running Phase 3 validation assertions...
──────────────────────────────────────────────────
✅ Split sizes sum correctly
✅ Single sample shape and dtype correct
✅ Batch shape correct
✅ Special tokens at correct indices (PAD=0, UNK=1)


✅ Label indices within valid range
✅ Vocabulary saves and reloads identically
✅ Vocabulary build-order check passed
──────────────────────────────────────────────────
🎉 All assertions passed!
   Vocab size : 23,262
   Classes    : 18
   MAX_LEN    : 256
   Batch size : 64
